# Notebook 0: What is Spark?

## Goal

The goal of this notebook is to understand Spark as a distributed data processing engine.

Spark is not just a "faster pandas." Spark is designed to process data across many machines. In small examples, it may feel similar to pandas because both can work with rows and columns. The key difference is that Spark is built around distributed computation.

In Spark, you write logical instructions such as:

- Read this data
- Select these columns
- Filter these rows
- Group by this column
- Compute this summary

Spark then decides how to execute those instructions across a cluster.

In this notebook, we will learn the basic mental model of Spark and create our first Spark DataFrame.


## Core Spark Concepts

| Concept | What You Should Understand |
|---|---|
| Spark | A distributed engine for processing data at scale |
| Driver | The process that coordinates the Spark job |
| Executors | Worker processes that run tasks |
| Cluster | The collection of compute resources Spark uses |
| SparkSession | Your main entry point into Spark from PySpark |
| DataFrame | Spark's main tabular data abstraction |
| Lazy evaluation | Spark waits to execute until an action is called |
| Transformations | Operations that define a new DataFrame |
| Actions | Operations that trigger execution |

Spark DataFrames are organized into named columns, similar to SQL tables or pandas DataFrames, but with distributed execution and query optimization underneath.

Reference for deeper learning:

https://spark.apache.org/docs/latest/sql-programming-guide.html


## 1. The Spark Mental Model

A Spark application has a few major parts:

### Driver

The driver is the coordinator. It runs your main program, keeps track of the operations you asked Spark to perform, and sends work to executors.

In a Databricks notebook, your notebook code interacts with the Spark driver.

### Executors

Executors are worker processes. They perform the actual computation on chunks of data.

For example, if you ask Spark to filter a billion rows, Spark can split that work across multiple executors.

### Cluster

A cluster is the collection of machines or compute resources available to Spark.

In Databricks Free Edition, you do not need to manually manage all of this complexity right away. But the mental model still matters because Spark code is designed for distributed execution.

### SparkSession

The `SparkSession` is your main entry point into Spark when using PySpark.

In Databricks notebooks, a SparkSession named `spark` is usually created for you automatically.


In [0]:
# In Databricks, the SparkSession is usually already available as a variable named spark.
# Let's inspect it.

spark


In [0]:
# We can check the type of the SparkSession object.
# This tells us what kind of Python object "spark" is.

type(spark)


pyspark.sql.connect.session.SparkSession

In [0]:
# We can also print useful information about the Spark version.
# This is helpful because Spark features can vary slightly by version.

print("Spark version:", spark.version)


Spark version: 4.1.0


## 2. Creating Your First Spark DataFrame

A Spark DataFrame is a distributed table-like object.

It has:

- Rows
- Columns
- Column names
- Column data types
- A schema

This makes it feel similar to a pandas DataFrame or SQL table.

However, the important difference is that a Spark DataFrame is designed to scale across a cluster.


In [0]:
# Create a small Python list of tuples.
# Each tuple represents one row of data.

retail_data = [
   (1, "Store_A", "Electronics", 249.99),
   (2, "Store_A", "Grocery", 34.50),
   (3, "Store_B", "Clothing", 89.99),
   (4, "Store_B", "Electronics", 599.99),
   (5, "Store_C", "Grocery", 22.25)
]

# Define column names for the DataFrame.
columns = ["transaction_id", "store_id", "category", "amount"]

# Create a Spark DataFrame from the Python list.
# Spark converts the local Python data into a distributed Spark DataFrame.

transactions_df = spark.createDataFrame(retail_data, columns)


In [0]:
# Display the DataFrame using Databricks' display() function.
# display() gives you a nice interactive table in Databricks.

display(transactions_df)


transaction_id,store_id,category,amount
1,Store_A,Electronics,249.99
2,Store_A,Grocery,34.5
3,Store_B,Clothing,89.99
4,Store_B,Electronics,599.99
5,Store_C,Grocery,22.25


In [0]:
# show() is the standard Spark method for printing rows of a DataFrame.
# It works in Databricks and outside of Databricks.

transactions_df.show()


+--------------+--------+-----------+------+
|transaction_id|store_id|   category|amount|
+--------------+--------+-----------+------+
|             1| Store_A|Electronics|249.99|
|             2| Store_A|    Grocery|  34.5|
|             3| Store_B|   Clothing| 89.99|
|             4| Store_B|Electronics|599.99|
|             5| Store_C|    Grocery| 22.25|
+--------------+--------+-----------+------+



In [0]:
# printSchema() shows the structure of the DataFrame.
# The schema tells us the column names and data types.

transactions_df.printSchema()


root
 |-- transaction_id: long (nullable = true)
 |-- store_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)



In [0]:
# The columns attribute gives us a Python list of column names.

transactions_df.columns


['transaction_id', 'store_id', 'category', 'amount']

In [0]:
# dtypes gives us the column names and their inferred Spark data types.

transactions_df.dtypes


[('transaction_id', 'bigint'),
 ('store_id', 'string'),
 ('category', 'string'),
 ('amount', 'double')]

## 3. Spark DataFrames vs pandas DataFrames

If you have used pandas before, Spark DataFrames may look familiar.

Both can represent tabular data.

But there are major differences:

| Topic | pandas | Spark |
|---|---|---|
| Execution | Runs locally on one machine | Can run across a cluster |
| Data size | Best for small to medium data | Designed for large-scale data |
| Evaluation | Usually eager | Lazy |
| Optimization | Mostly manual | Uses Spark's query optimizer |
| Common use | Analysis on local data | Distributed ETL, analytics, ML pipelines |

The most important difference for now is lazy evaluation.

In pandas, when you run a line of code, the result is usually computed immediately.

In Spark, many operations do not execute immediately. Spark builds a plan and waits until you call an action.


In [0]:
# Optional: Create a small pandas DataFrame for comparison.
# This is only for conceptual comparison.
# pandas runs locally in memory on the driver machine.

import pandas as pd

pandas_df = pd.DataFrame(
   retail_data,
   columns=columns
)

pandas_df


,transaction_id,store_id,category,amount
0,1,Store_A,Electronics,249.99
1,2,Store_A,Grocery,34.50
2,3,Store_B,Clothing,89.99
3,4,Store_B,Electronics,599.99
4,5,Store_C,Grocery,22.25


In [0]:
# Check the type of the pandas DataFrame.

type(pandas_df)

pandas.core.frame.DataFrame

In [0]:
# Check the type of the Spark DataFrame.

type(transactions_df)


pyspark.sql.connect.dataframe.DataFrame

## 4. Transformations

A transformation defines a new DataFrame from an existing DataFrame.

Examples of transformations include:

- `select()`
- `filter()`
- `where()`
- `withColumn()`
- `groupBy()`
- `orderBy()`

Transformations are lazy.

That means Spark does not immediately execute the operation. Instead, it records the operation as part of a logical plan.


In [0]:
# This is a transformation.
# We are defining a new DataFrame containing only high-value transactions.
# Spark does NOT execute the filter immediately.

high_value_df = transactions_df.filter(transactions_df.amount > 100)

# Notice that this cell does not display any rows yet.
# We have only defined the transformation.


In [0]:
# Let's inspect the type of high_value_df.
# It is another Spark DataFrame.

type(high_value_df)


pyspark.sql.connect.dataframe.DataFrame

## 5. Actions

An action triggers Spark to actually execute the computation.

Examples of actions include:

- `show()`
- `display()`
- `count()`
- `collect()`
- `first()`
- `take()`
- writing data to storage

Until you call an action, Spark is mostly building a plan.

When you call an action, Spark sends work to the executors and computes the result.


In [0]:
# show() is an action.
# This triggers Spark to execute the filter transformation defined earlier.

high_value_df.show()


+--------------+--------+-----------+------+
|transaction_id|store_id|   category|amount|
+--------------+--------+-----------+------+
|             1| Store_A|Electronics|249.99|
|             4| Store_B|Electronics|599.99|
+--------------+--------+-----------+------+



In [0]:
# count() is also an action.
# Spark must scan the DataFrame and count how many rows match.

num_high_value_transactions = high_value_df.count()

print("Number of high-value transactions:", num_high_value_transactions)


Number of high-value transactions: 2


## 6. Chaining Transformations

Spark code often chains multiple transformations together.

This lets you express a logical data pipeline.

In the example below, we will:

1. Filter for transactions over 50 dollars
2. Select only a few columns
3. Sort by amount in descending order

These transformations define a plan.

The plan executes when we call `show()` or `display()`.


In [0]:
# Chain multiple transformations together.
# This still does not execute until an action is called.

filtered_sorted_df = (
   transactions_df
   .filter(transactions_df.amount > 50)
   .select("transaction_id", "store_id", "category", "amount")
   .orderBy("amount", ascending=False)
)

# The variable filtered_sorted_df now points to a new Spark DataFrame.
# The computation has not been triggered yet.


In [0]:
# display() is an action in Databricks.
# This triggers Spark to execute the chain of transformations.

display(filtered_sorted_df)

transaction_id,store_id,category,amount
4,Store_B,Electronics,599.99
1,Store_A,Electronics,249.99
3,Store_B,Clothing,89.99


## 7. Inspecting the Query Plan with explain()

Spark does not simply run your code line by line like normal Python.

Instead, Spark builds a query plan.

The query plan describes how Spark intends to execute your transformations.

The `explain()` method lets us inspect that plan.

You do not need to understand every detail yet. For now, the key idea is:

> Spark converts your DataFrame operations into an execution plan.


In [0]:
# explain() shows the physical plan Spark will use.
# This is useful later when learning performance optimization.

filtered_sorted_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [amount#13275 DESC NULLS LAST]
         +- PhotonRowToColumnar
            +- LocalTableScan [transaction_id#13272L, store_id#13273, category#13274, amount#13275]


== Photon Explanation ==
The query is fully supported by Photon.


## 8. Aggregation Example

Spark is often used to summarize large datasets.

A common pattern is:

1. Group rows by one or more columns
2. Compute summary statistics

For example, we can calculate total sales by store.

In [0]:
from pyspark.sql import functions as F

# groupBy() defines grouped computation.
# agg() defines the summary calculation.
# This is a transformation until we call display() or show().

sales_by_store_df = (
   transactions_df
   .groupBy("store_id")
   .agg(
       F.sum("amount").alias("total_sales"),
       F.count("*").alias("num_transactions")
   )
   .orderBy("total_sales", ascending=False)
)

display(sales_by_store_df)


store_id,total_sales,num_transactions
Store_B,689.98,2
Store_A,284.49,2
Store_C,22.25,1


## 9. Registering a Temporary View

Spark lets us use both the DataFrame API and SQL.

To query a DataFrame with SQL, we can register it as a temporary view.

A temporary view exists only for the current Spark session.


In [0]:
# Register the Spark DataFrame as a temporary SQL view.
# This allows us to query the DataFrame using SQL syntax.

transactions_df.createOrReplaceTempView("transactions")


In [0]:
# Use Spark SQL to query the temporary view.
# The result of spark.sql() is also a Spark DataFrame.

sql_result_df = spark.sql("""
   SELECT
       store_id,
       SUM(amount) AS total_sales,
       COUNT(*) AS num_transactions
   FROM transactions
   GROUP BY store_id
   ORDER BY total_sales DESC
""")

display(sql_result_df)


store_id,total_sales,num_transactions
Store_B,689.98,2
Store_A,284.49,2
Store_C,22.25,1


## 10. DataFrame API vs Spark SQL

The DataFrame API and Spark SQL are two different ways to express similar logic.

For example, these two approaches are equivalent:

### DataFrame API

```python
transactions_df.groupBy("store_id").agg(F.sum("amount"))
```
### Spark SQL
```python
SELECT store_id, SUM(amount)
FROM transactions
GROUP BY store_id
```

In real projects, you may use both.
Python is often useful for building pipelines and reusable logic.
SQL is often useful for analytics, reporting, and communicating with business users.


## 11. Key Lesson

Spark is not "faster pandas."

Spark is a distributed computation engine.

You write logical operations. Spark builds a plan. Then Spark decides how to execute that plan across available compute resources.

The most important ideas from this notebook are:

1. Spark uses a driver and executors.
2. The `SparkSession` is your entry point into Spark.
3. A Spark DataFrame is a distributed table-like abstraction.
4. Transformations define new DataFrames.
5. Actions trigger execution.
6. Spark uses lazy evaluation.
7. Spark can be used with both Python syntax and SQL.

# Practice Exercises

# Practice Exercises

Try these on your own before looking for help.

## Exercise 1

Create a new Spark DataFrame called `employees_df` with the following columns:

- employee_id
- name
- department
- salary

Add at least 6 rows of data.

Then use:

- `display()`
- `show()`
- `printSchema()`

## Exercise 2

Filter the employee DataFrame to show only employees with salary greater than 75,000.

## Exercise 3

Group the employee DataFrame by department and calculate:

- average salary
- number of employees

## Exercise 4

Create a temporary view called `employees`.

Then write a SQL query to calculate average salary by department.

## Exercise 5

Use `explain()` on one of your filtered or grouped DataFrames.


# Optional Answer Starter for Exercises

In [0]:
# Exercise starter code
# Try modifying this with your own values.

employee_data = [
   (1, "Alice", "Data Science", 95000),
   (2, "Bob", "Engineering", 88000),
   (3, "Carlos", "Engineering", 72000),
   (4, "Diana", "Operations", 68000),
   (5, "Evan", "Data Science", 105000),
   (6, "Fatima", "Operations", 76000)
]

employee_columns = ["employee_id", "name", "department", "salary"]

employees_df = spark.createDataFrame(employee_data, employee_columns)

display(employees_df)

# Filter employees with salary greater than 75,000.
# This is a transformation until display() is called.

high_salary_df = employees_df.filter(employees_df.salary > 75000)

display(high_salary_df)

# Group by department and calculate average salary and employee count.

salary_by_department_df = (
   employees_df
   .groupBy("department")
   .agg(
       F.avg("salary").alias("avg_salary"),
       F.count("*").alias("num_employees")
   )
   .orderBy("avg_salary", ascending=False)
)

display(salary_by_department_df)

# Register employees_df as a temporary SQL view.

employees_df.createOrReplaceTempView("employees")

# Query the employees view using Spark SQL.

employee_sql_result_df = spark.sql("""
   SELECT
       department,
       AVG(salary) AS avg_salary,
       COUNT(*) AS num_employees
   FROM employees
   GROUP BY department
   ORDER BY avg_salary DESC
""")

display(employee_sql_result_df)

# Inspect the query plan.

employee_sql_result_df.explain()


# Final Takeaways

By the end of this first notebook, you should understand the basic Spark workflow:
```
Create DataFrame
   ↓
Apply transformations
   ↓
Call an action
   ↓
Spark executes the plan
The core mental model is:
You define what should happen.
Spark decides how to execute it.
```
